1 - bibliotecas e importação dos dados:

In [ ]:
import pandas as pd

df = pd.read_csv('data/desafio_transacoes_financeras.csv')

2 - analise dos dados:

2.1 - analise das 5 primeiras linhas

In [ ]:
df.head(5)

2.2 - analise do formato do dataset

In [ ]:
df.shape

2.3 - analise dos tipos de dados

In [ ]:
df.dtypes

nota: todaas as colunas foram consideradas object. e todas devem ser mudadas para os seguinte tipos:<br>
    1 - transaction_id --> String<br>
    2 - date --> datetime64<br>
    3 - description --> String<br>
    4 - amount --> float64<br>
    5 - category --> category<br>
    6 - status --> category<br>
    7- account --> category

2.4 - analise de dados vazios:

2.4.1 - analise da somatoria de numeros vazios de cada coluna

In [ ]:
df.isnull().sum()

2.4.2 - analise da porcentagem de valores vazios em relação a tabela

In [ ]:
(df.isnull().sum() / len(df)) * 100

2.4.3 - analise das colunas vazias

In [ ]:
df[df["amount"].isnull()] # codigo para verificar as linhas vazias da coluna amount

nota: ao analisar o dataset, percebi que a linha 5 e 6 são clones, e pelo fato de terem valores vazios importantes que podem atrapalhar a analise, irei remover ambas linhas

In [ ]:
df[df["date"].isnull()] # codigo para verificar as linhas vazias da coluna amount

nota: a linha 17 esta extremamente problematica, por conta disso, irei remover ela tambem

2.5 - analise de duplicatas

In [ ]:
duplicadas_completas = df[df.duplicated(subset=['date', 'description', 'amount', 'category', 'status', 'account'], keep=False)]
display(duplicadas_completas)

nota: como dito anteriormente, a linha 5 e 6 serão removidas devido o valor vazio, e a linha 14 será removida devido a duplicata em relação a linha 13

2.6 - analise dos valores unicos

na analise, irei focar apenas nas colunas mais importantes, como a date, amount, status

In [ ]:
df["date"].unique()

nota: varias problemas de formatação nas datas. na formatação, irei usar o padrão ISO 8601: AA-MM-DD

In [ ]:
df["amount"].unique()

nota: tambem contem varios problemas de formatação. como todo o dataset e em ingles, presumo que os valores são descritos da seguinte forma: 00.00

In [ ]:
df["status"].unique()

nota: deve-se corrigir a palavra completed, para que siga o padrão.

--------------------------------------------------------------------------------------------------------------

3 - correção dos dados:

obs: criarei uma copia para não correr riscos de perder os dados originais

In [ ]:
df_clean = df.copy()

In [ ]:
df_clean.head(5)

3.1 - remoção das duplicatas

como dito antes, irei remover as linhas 5 e 6 devido a duplicata e valor nulo na coluna amount. a linha 14 devido ser duplicata da linha 13 e a linha 17 devido ela estar completamente errada

In [ ]:
df_clean = df_clean.drop([5,6,14,17])

3.2 - formatação da coluna date:

3.2.1 - troca dos simbolos"/" para "-"

In [ ]:
df_clean["date"] = (df_clean["date"].str.replace("/", "-", regex=False))

3.2.2 - padronização do dataset

In [ ]:
df_clean["date"] = pd.to_datetime( # transforma dados de string e numericos e valores de data
    df_clean["date"],
    format="mixed",    # usei para indicar ao pandas que na coluna há varios formatos
    dayfirst=True,     # usei para indicar ao pandas o formato padrão
    errors="coerce"    # apenas para substituir valores de data bugados para NaT
)

3.3 - formatação da coluna amount:

3.3.1 - remoção de espaços em branco

In [ ]:
df_clean["amount"] = df_clean["amount"].str.strip()

3.3.2 - variavel para salvar os valores negativos para ser reutilizado depois

In [ ]:
is_negative = df_clean["amount"].str.contains("-")

3.3.3 - remoção de simbolos para formatação adequada

In [ ]:
df_clean["amount"] = df_clean["amount"].str.replace(r"[^\d,.]", "", regex=True)

3.3.4 - substituição de "," para "."

In [ ]:
df_clean["amount"] = df_clean["amount"].str.replace(",", ".")

3.3.5 - transformação dos valores em float

In [ ]:
df_clean["amount"] = pd.to_numeric(df_clean["amount"], errors="coerce")

3.3.6 - recuperação dos valores negativos

In [ ]:
df_clean.loc[is_negative, "amount"] = (df_clean.loc[is_negative, "amount"] * -1)

In [ ]:
df_clean.head(10)

3.4 - formatação da coluna status

In [ ]:
df_clean["status"] = df_clean["status"].replace({"completed": "Completed"})

3.5 modificação de cada coluna para seu determinado tipo

nota: data, amount ja foram modificadas durante a formatação

In [ ]:
tipos_colunas = {
    "transaction_id": "string",
    "date": "datetime64[ns]",
    "description": "string",
    "category": "category",
    "status": "category",
    "account": "category",
}
df_clean = df_clean.astype(tipos_colunas)

---------------------------------------------------------------------------------------------------

4 - salvamento para um novo dataset

4.1 - salvamento em csv

In [ ]:
df_clean.to_csv("data/desafio_transacoes_limpo.csv", index=False, encoding="utf-8")

4.2 - salvamento em excel

In [ ]:
df_clean.to_excel("data/desafio_transacoes_limpo.xlsx", index=False)

In [ ]:
df_clean.head(5)